# Section III: Persona Data Generation

**Talk section**: Section III — The Proposed Framework

This notebook walks through the design and execution of the data generation pipeline for trader personas. It is the first step in the three-step framework: **data generation → LoRA fine-tuning → simulation and evaluation**.

By the end of this notebook you will understand:
- What persona-conditioned training data is and why we need it
- The behavioral finance theory behind each of the three personas
- How we prompt `claude-sonnet-4-6` to generate reasoning traces
- What the generated data looks like and how it differs across personas

## What Is Persona-Conditioned Training Data?

A small language model (SLM) by itself has no particular trading personality. If you ask it to make a trading decision, it will produce a generic, averaged response drawn from its pretraining distribution — not a coherent persona.

To create **behavioral heterogeneity**, we need each persona to reason in a systematically different way:
- A **momentum trader** should look at price trends and extrapolate them forward
- A **value investor** should compare the current price to an estimate of fair value
- A **noise trader** should react to news and sentiment in a partially random way

We achieve this through **supervised fine-tuning** on persona-specific examples. Each example is a chain-of-thought reasoning trace that shows the model *how to think* about a market state from a particular persona's perspective.

The training data is generated by prompting a large, capable model (Claude) with the persona's identity and a market scenario. Claude, having been trained on vast amounts of financial text, can write convincing persona-appropriate reasoning. We then use these traces to fine-tune the smaller model.

This is **persona distillation**: knowledge transfer from a large model to a small one, mediated by behavioral personas.

In [1]:
# Imports and setup
import json
import os
import sys
from pathlib import Path

# Add the project root to the path so we can import from data/
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print(f"Python version: {sys.version}")

# Check for API key
api_key = os.environ.get('ANTHROPIC_API_KEY')
if api_key:
    print("ANTHROPIC_API_KEY is set.")
else:
    print("WARNING: ANTHROPIC_API_KEY is not set. API calls will fail.")
    print("Set it with: export ANTHROPIC_API_KEY=your_key_here")

Project root: /Users/joelfuentes/Documents/claude_code_projects/ucfinmath
Python version: 3.14.3 | packaged by Anaconda, Inc. | (main, Feb 24 2026, 22:43:19) [Clang 20.1.8 ]
ANTHROPIC_API_KEY is set.


## The Three Personas

Each persona is grounded in a specific strand of behavioral finance theory. The prompt identity strings are used both during data generation (as a system prompt to Claude) and during simulation inference (as a system prompt to the fine-tuned SLM). Train/inference consistency is critical.

### Momentum Trader
**Theory**: Jegadeesh & Titman (1993) documented that stocks that outperformed over the past 3–12 months tend to continue outperforming. Momentum traders exploit this by following trends, ignoring fundamentals entirely.

### Value Investor
**Theory**: Kahneman & Tversky's prospect theory and Shefrin & Statman's behavioral portfolio theory show that investors anchor to reference points. Value investors anchor to *intrinsic value*: they buy when prices fall below fair value and sell when they rise above it, providing a stabilizing mean-reversion force.

### Noise Trader
**Theory**: De Long, Shleifer, Summers & Waldmann (1990) showed that noise traders — who trade on sentiment rather than information — can survive in equilibrium and affect prices. Odean (1999) documented systematic overconfidence in retail investors. Noise traders add entropy and amplify sentiment-driven moves.

In [2]:
# The persona identity strings — used in both data generation and simulation
# These are the system prompts that define each persona's behavioral identity

PERSONA_IDENTITIES = {
    "momentum": (
        "You are a momentum trader. You believe recent price trends persist in the short run. "
        "You buy assets that have risen recently and sell assets that have fallen. "
        "You do not anchor to fundamentals."
    ),
    "value": (
        "You are a value investor. You estimate the intrinsic value of an asset from its "
        "fundamentals. You buy when the price is significantly below fair value and sell when "
        "it exceeds it. You are not influenced by short-term price momentum."
    ),
    "noise": (
        "You are a noise trader. You react to market news and social sentiment. Your decisions "
        "are influenced by recent headlines and what you believe other traders are thinking. "
        "You trade frequently and are prone to overconfidence."
    ),
}

for persona, identity in PERSONA_IDENTITIES.items():
    print(f"--- {persona.upper()} ---")
    print(identity)
    print()

--- MOMENTUM ---
You are a momentum trader. You believe recent price trends persist in the short run. You buy assets that have risen recently and sell assets that have fallen. You do not anchor to fundamentals.

--- VALUE ---
You are a value investor. You estimate the intrinsic value of an asset from its fundamentals. You buy when the price is significantly below fair value and sell when it exceeds it. You are not influenced by short-term price momentum.

--- NOISE ---
You are a noise trader. You react to market news and social sentiment. Your decisions are influenced by recent headlines and what you believe other traders are thinking. You trade frequently and are prone to overconfidence.



## The Prompt Template

Each API call consists of:
1. **System prompt**: The persona identity string (above)
2. **User prompt**: The market state description + instructions to produce a JSON decision

We ask the model to produce:
- A **reasoning chain** (2-4 sentences of in-character thinking)
- An **action** (BUY, SELL, or HOLD)
- A **quantity** (1-20 shares, or 0 for HOLD)

The prompt format is designed to mirror exactly what will be used during simulation inference. This train/inference consistency is essential for the fine-tuned adapter to generalize.

In [3]:
# Show the prompt template with a concrete example market state
# This is the exact format used in data/generate_personas.py

example_market_state = {
    "price_history": [100.0, 102.0, 105.0, 107.0, 110.0],
    "news": "Strong earnings report released this morning.",
    "fair_value": 104.0,
}

def build_user_prompt(market_state):
    prices = market_state['price_history']
    current_price = prices[-1]
    price_change = round((current_price - prices[0]) / prices[0] * 100, 2)
    return (
        f"You are analyzing a financial market. Here is the current market state:\n\n"
        f"Price history (last 5 periods): {prices}\n"
        f"Current price: {current_price}\n"
        f"Price change over window: {price_change}%\n"
        f"Latest news: {market_state['news']}\n"
        f"Estimated fair value: {market_state['fair_value']}\n\n"
        f"Based on your identity as a trader, reason step-by-step about what you observe "
        f"and what action you will take.\n\n"
        f"Then provide your decision in exactly this JSON format (and nothing else after it):\n"
        '{\n  \"reasoning\": \"<your step-by-step reasoning, 2-4 sentences>\",\n'
        '  \"action\": \"<BUY or SELL or HOLD>\",\n'
        '  \"quantity\": <integer from 1 to 20, or 0 if HOLD>\n}\n\n'
        "Remember: you must reason in character. Do not break character or hedge."
    )

prompt = build_user_prompt(example_market_state)
print("=== USER PROMPT ===")
print(prompt)

=== USER PROMPT ===
You are analyzing a financial market. Here is the current market state:

Price history (last 5 periods): [100.0, 102.0, 105.0, 107.0, 110.0]
Current price: 110.0
Price change over window: 10.0%
Latest news: Strong earnings report released this morning.
Estimated fair value: 104.0

Based on your identity as a trader, reason step-by-step about what you observe and what action you will take.

Then provide your decision in exactly this JSON format (and nothing else after it):
{
  "reasoning": "<your step-by-step reasoning, 2-4 sentences>",
  "action": "<BUY or SELL or HOLD>",
  "quantity": <integer from 1 to 20, or 0 if HOLD>
}

Remember: you must reason in character. Do not break character or hedge.


## What We Ask the LLM to Generate

For each API call, we give Claude:
- A persona identity as the system prompt
- A market state (price history, news, fair value) as the user prompt
- Instructions to reason step-by-step and produce a JSON decision

We validate the output before saving:
- `action` must be one of BUY, SELL, HOLD
- `quantity` must be an integer in [0, 20]
- The JSON must be parseable

Malformed outputs are logged and skipped. We aim for ~300 valid examples per persona, covering the range of market scenarios (uptrends, downtrends, price above/below fair value, positive/negative/neutral news).

The full generation script is in `data/generate_personas.py`. Below, we demonstrate a single API call.

In [4]:
# Demo: generate one example using the Anthropic API
# Requires ANTHROPIC_API_KEY to be set

# If you don't have an API key, skip this cell and load the pre-generated data below

import anthropic

def generate_one_example(persona, market_state):
    """Call the API and return one training example."""
    api_key = os.environ.get('ANTHROPIC_API_KEY')
    if not api_key:
        print("Skipping: ANTHROPIC_API_KEY not set.")
        return None
    
    client = anthropic.Anthropic(api_key=api_key)
    system_prompt = PERSONA_IDENTITIES[persona]
    user_prompt = build_user_prompt(market_state)
    
    message = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=512,
        system=system_prompt,
        messages=[{"role": "user", "content": user_prompt}],
    )
    raw_text = message.content[0].text.strip()
    
    # Parse the JSON block
    json_start = raw_text.rfind('{')
    json_end = raw_text.rfind('}') + 1
    parsed = json.loads(raw_text[json_start:json_end])
    
    return {
        "persona": persona,
        "market_state": market_state,
        "reasoning": parsed["reasoning"],
        "action": parsed["action"],
        "quantity": parsed["quantity"],
    }

# Try generating one example for the momentum persona
result = generate_one_example("momentum", example_market_state)
if result:
    print(f"Persona: {result['persona']}")
    print(f"Reasoning: {result['reasoning']}")
    print(f"Action: {result['action']}, Quantity: {result['quantity']}")
else:
    print("API call skipped. Loading from pre-generated data instead.")

Persona: momentum
Reasoning: The price has risen consistently across all 5 periods with no pullbacks, reflecting strong and accelerating upward momentum. The strong earnings news this morning is a fresh catalyst likely to attract more buyers and sustain the trend. As a momentum trader, I ignore the fair value anchor — the trend is my signal, and it's firmly bullish. I buy aggressively to capture the continuation.
Action: BUY, Quantity: 15


## Comparing Personas: Same Market State, Different Reasoning

The most compelling way to understand what the data generation achieves is to see all three personas respond to *the same market state*. The market state we use:

- **Price history**: [100, 102, 105, 107, 110] — a clear uptrend (+10%)
- **News**: "Strong earnings report released this morning." — positive
- **Fair value**: 104 — current price is *above* fair value

This is a scenario where the three personas should strongly disagree:
- **Momentum**: The trend is up, news is good → BUY
- **Value**: Price is above fair value → SELL (or at least not buy)
- **Noise**: Reacts to positive news and trend → probably BUY, but erratically

The side-by-side comparison below makes the behavioral divergence concrete.

In [5]:
# Load the first few examples from each persona's JSONL file
# This demonstrates the pre-generated placeholder data

def load_jsonl(path, n=3):
    """Load the first n examples from a .jsonl file."""
    examples = []
    filepath = project_root / path
    if not filepath.exists():
        print(f"File not found: {filepath}")
        return []
    with open(filepath) as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            if line.strip():
                examples.append(json.loads(line))
    return examples

personas = ["momentum", "value", "noise"]

for persona in personas:
    print(f"\n{'='*60}")
    print(f"PERSONA: {persona.upper()}")
    print('='*60)
    examples = load_jsonl(f"data/personas/{persona}.jsonl", n=2)
    for i, ex in enumerate(examples):
        print(f"\n  Example {i+1}:")
        print(f"  Market state: price={ex['market_state']['price_history']}, "
              f"fv={ex['market_state']['fair_value']}")
        print(f"  News: {ex['market_state']['news']}")
        print(f"  Reasoning: {ex['reasoning']}")
        print(f"  >>> ACTION: {ex['action']}, QUANTITY: {ex['quantity']}")


PERSONA: MOMENTUM

  Example 1:
  Market state: price=[100, 102, 105, 107, 110], fv=104
  News: Strong earnings report released this morning.
  Reasoning: Prices have risen 10% over 5 periods with no reversal. The trend is intact. Earnings news confirms the bullish narrative. I will buy.
  >>> ACTION: BUY, QUANTITY: 10

  Example 2:
  Market state: price=[115, 112, 109, 106, 103], fv=110
  News: Analyst downgrades sector outlook.
  Reasoning: Prices have fallen roughly 10% over 5 periods. The downtrend shows no sign of reversing. The analyst downgrade reinforces the bearish momentum. I will sell.
  >>> ACTION: SELL, QUANTITY: 8

PERSONA: VALUE

  Example 1:
  Market state: price=[100, 102, 105, 107, 110], fv=104
  News: Strong earnings report released this morning.
  Reasoning: Current price is 110, which is 5.8% above my estimated fair value of 104. The asset is overextended. Strong earnings may justify a slight premium, but not this much. I will sell into this strength.
  >>> ACTION

In [6]:
# Find examples from the same market scenario across all three personas
# to show side-by-side comparison

# Load all examples from each persona
all_examples = {}
for persona in personas:
    all_examples[persona] = load_jsonl(f"data/personas/{persona}.jsonl", n=5)

# The first example in each file uses the same uptrend market state
print("SIDE-BY-SIDE: Same uptrend scenario, three personas")
print(f"Market state: {all_examples['momentum'][0]['market_state']}")
print()

for persona in personas:
    if all_examples[persona]:
        ex = all_examples[persona][0]
        print(f"[{persona.upper()}]")
        print(f"  Reasoning: {ex['reasoning']}")
        print(f"  Decision:  {ex['action']} {ex['quantity']} shares")
        print()

SIDE-BY-SIDE: Same uptrend scenario, three personas
Market state: {'price_history': [100, 102, 105, 107, 110], 'news': 'Strong earnings report released this morning.', 'fair_value': 104}

[MOMENTUM]
  Reasoning: Prices have risen 10% over 5 periods with no reversal. The trend is intact. Earnings news confirms the bullish narrative. I will buy.
  Decision:  BUY 10 shares

[VALUE]
  Reasoning: Current price is 110, which is 5.8% above my estimated fair value of 104. The asset is overextended. Strong earnings may justify a slight premium, but not this much. I will sell into this strength.
  Decision:  SELL 5 shares

[NOISE]
  Reasoning: Earnings beat! Everyone is going to be buying this. The price is already moving up and I don't want to miss the wave. Other traders are clearly piling in. I need to get in now before it goes even higher.
  Decision:  BUY 13 shares



## Summary and Next Steps

In this notebook we've seen:

1. **What persona-conditioned data is**: Each training example is a (market state, reasoning chain, action) triple where the reasoning is written from the perspective of a specific behavioral persona.

2. **The prompt design**: We use the persona identity as a system prompt and the market state as a user prompt, asking for structured JSON output. This format is used consistently in training and at inference time.

3. **The behavioral divergence**: Even with identical inputs, the three personas produce fundamentally different reasoning and often conflicting decisions. Momentum buys a rising price; value sells it (if above fair value); noise follows sentiment erratically.

4. **The full generation script**: `data/generate_personas.py` can generate ~300 examples per persona using the Anthropic API. Run with `--export-examples` to produce the cross-persona comparison file for the Streamlit app.

**Next**: Notebook `02_finetuning_walkthrough.ipynb` shows how we use this data to fine-tune LoRA adapters on top of `Qwen/Qwen2.5-1.5B-Instruct`.